# Paced latency, CUDA backend (rerun after the tied-tail fix)

In [ ]:
# Kaggle: GPU T4, internet on, private dataset rajulkabir/tileproof-src-warps attached
# (triton_equiv and tinytail source as a tarball; SOURCE.txt names the commits).
import glob, os, subprocess, sys, tarfile
os.environ["TRITON_INTERPRET"] = "0"
SRC = "/kaggle/working/tileproof_src"
import shutil
tars = glob.glob("/kaggle/input/**/tileproof_src.tar.gz", recursive=True)
srcs = glob.glob("/kaggle/input/**/SOURCE.txt", recursive=True)
os.makedirs(SRC, exist_ok=True)
if tars:
    with tarfile.open(tars[0]) as t:
        t.extractall(SRC)
elif srcs:  # Kaggle unpacked the tarball
    shutil.copytree(os.path.dirname(srcs[0]), SRC, dirs_exist_ok=True)
else:
    raise SystemExit("dataset missing: " + str(glob.glob("/kaggle/input/**", recursive=True)[:50]))
print(open(SRC + "/SOURCE.txt").read())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver", "ninja", SRC + "/tinytail"], check=True)
sys.path[:0] = [SRC]

In [ ]:
import json, torch
print(torch.__version__, torch.cuda.get_device_name(0))
from triton_equiv.cuda_ref import Backend
from triton_equiv.msync import cuda_vm, latency
print(cuda_vm.build_error(Backend.CUDA) or "CUDA VM built")
lat_pc = latency.main(["--backend", "cuda", "--design", "paced", "--n", "100000",
                       "--out", "/kaggle/working/msync_gpu_latency_paced_cuda.json"])
print(json.dumps(lat_pc["paired"], indent=1))
for r in lat_pc["rows"]:
    print(r["program"], r["schedule"], round(r["median_us"], 2), r["gate_verdict"], r["primary_outcome"])